In [24]:
from google.colab import files
files.upload()

!pip install scikit-fuzzy

import numpy as np
import pandas as pd
import skfuzzy as fuzz
from skfuzzy import control as ctrl

Saving mammographic.dat to mammographic (3).dat


In [25]:
rows = []

with open("mammographic.dat", "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        parts = line.replace(",", " ").split()
        if len(parts) == 6:
            rows.append(parts)

data = pd.DataFrame(
    rows,
    columns=["BI-RADS", "Age", "Shape", "Margin", "Density", "Severity"]
)

# Convert to numeric and remove invalid rows
data = data.apply(pd.to_numeric, errors="coerce")
data = data.dropna()

data.head()

,BI-RADS,Age,Shape,Margin,Density,Severity
1,5.0,67.0,3.0,5.0,3.0,1.0
3,5.0,58.0,4.0,5.0,3.0,1.0
4,4.0,28.0,1.0,1.0,3.0,0.0
9,5.0,57.0,1.0,5.0,3.0,1.0
11,5.0,76.0,1.0,4.0,3.0,1.0


In [26]:
X = data[["Age", "Margin", "Density"]]
y = data["Severity"]

In [27]:
X_norm = (X - X.min()) / (X.max() - X.min())
X_norm.head()

,Age,Margin,Density
1,0.628205,1.00,0.666667
3,0.512821,1.00,0.666667
4,0.128205,0.00,0.666667
9,0.500000,1.00,0.666667
11,0.743590,0.75,0.666667


In [28]:
X_benign = X_norm[y == 0]
X_malignant = X_norm[y == 1]

In [29]:
stats = {
    "benign": {
        "mean": X_benign.mean(),
        "std": X_benign.std()
    },
    "malignant": {
        "mean": X_malignant.mean(),
        "std": X_malignant.std()
    }
}

stats

{'benign': {'mean': Age        0.401249
  Margin     0.234778
  Density    0.630757
  dtype: float64,
  'std': Age        0.175675
  Margin     0.346092
  Density    0.126182
  dtype: float64},
 'malignant': {'mean': Age        0.572469
  Margin     0.684864
  Density    0.646816
  dtype: float64,
  'std': Age        0.158554
  Margin     0.291794
  Density    0.105894
  dtype: float64}}

In [30]:
def gaussian_mf(x, mean, std):
    std = std if std > 1e-6 else 1e-6
    return np.exp(-((x - mean) ** 2) / (2 * std ** 2))

In [31]:
def firing_strength(sample, class_stats):
    strengths = []
    for feature in sample.index:
        mu = gaussian_mf(
            sample[feature],
            class_stats["mean"][feature],
            class_stats["std"][feature]
        )
        strengths.append(mu)
    return np.prod(strengths)  # AND operator (product)

In [36]:
predictions = []

for _, sample in X_norm.iterrows():
    fs_benign = firing_strength(sample, stats["benign"])
    fs_malignant = firing_strength(sample, stats["malignant"])

    pred = 1 if fs_malignant > fs_benign else 0
    predictions.append(pred)

In [35]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

cm = confusion_matrix(y, predictions)
accuracy = accuracy_score(y, predictions)

print("Confusion Matrix:\n", cm)
print("\nAccuracy:", accuracy)
print("\nClassification Report:\n")
print(classification_report(y, predictions, target_names=["Benign", "Malignant"]))

Confusion Matrix:
 [[327 100]
 [ 88 315]]

Accuracy: 0.7734939759036145

Classification Report:

              precision    recall  f1-score   support

      Benign       0.79      0.77      0.78       427
   Malignant       0.76      0.78      0.77       403

    accuracy                           0.77       830
   macro avg       0.77      0.77      0.77       830
weighted avg       0.77      0.77      0.77       830

